In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
# Question 1
len(documents)

72

In [5]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [6]:
# Question 4
len(chunks)

295

In [4]:
documents[0].keys()

dict_keys(['content', 'filename'])

In [5]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [6]:
# Question 2
query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query,
    num_results=1
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [7]:
from rag_helper import RAGBase

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

rag = RAGBase(index, openai_client)

RAGBase initialized with model: gpt-5.4-mini


In [10]:
response = rag.rag("How does the agentic loop keep calling the model until it stops?")

The loop keeps calling the model with a `while True` loop, and after each response it checks whether any `function_call` items were returned.

- If there are function calls, it runs the tool, appends the tool result to the message history, and calls the model again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is simply: **no function calls this turn**. 7136


In [ ]:
# Question 3
response

['The loop keeps calling the model with a `while True` loop, and after each response it checks whether any `function_call` items were returned.\n\n- If there are function calls, it runs the tool, appends the tool result to the message history, and calls the model again.\n- If there are no function calls, it breaks out of the loop and stops.\n\nSo the stop condition is simply: **no function calls this turn**.',
 7136]

In [15]:

chunked_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunked_index.fit(chunks)

In [ ]:
rag_with_chunks = RAGBase(chunked_index, openai_client)
response = rag.rag("How does the agentic loop keep calling the model until it stops?")

RAGBase initialized with model: gpt-5.4-mini
It keeps calling the model inside a `while True` loop. After each model response, the code checks whether there were any `function_call` items:

- If there is a function call, it runs the tool, appends the tool output, and loops again.
- If there are no function calls, it `break`s out of the loop.

So the stop condition is simply: **no function calls in the latest response**. 7136


In [ ]:
# Question 5
response

['It keeps calling the model inside a `while True` loop. After each model response, the code checks whether there were any `function_call` items:\n\n- If there is a function call, it runs the tool, appends the tool output, and loops again.\n- If there are no function calls, it `break`s out of the loop.\n\nSo the stop condition is simply: **no function calls in the latest response**.',
 7136]

In [7]:
from minsearch import Index

chunked_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunked_index.fit(chunks)

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the chunked index for the given query and return the results as a list of dictionaries.
    """
    search_results = chunked_index.search(
        query,
        num_results=5
    )
    return search_results

In [9]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [10]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [12]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the chunked index for the given query and return the results as a list of dictionaries.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [13]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [14]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

In [15]:

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
# Question 6
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
